<h2>Imports</h2>

In [ ]:
import tensorflow as tf
import os
import numpy as np
from matplotlib import pyplot as plt
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, BatchNormalization, Dropout, Flatten, Dense, MaxPool2D, GlobalAveragePooling2D, Input, Layer, Concatenate, Add
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import ResNet50 , MobileNetV2 , VGG16 , InceptionV3 , Xception
import pandas as pd
import time

from utils.utils import plot_metrics

GPU Check

In [ ]:
tf.config.list_physical_devices("GPU")

<h2>Load Data

Depending on the models being trained, use one to load the dataset in the required image size

Image size 256x256 (CNN)

In [ ]:
data256 = tf.keras.utils.image_dataset_from_directory('CommonVoice/mfcc/mfcc_images' , batch_size=32, image_size=(256,256))
data256 = data256.map(lambda x,y: (x/255, y))

Image size 224x224 (ResNet50, MobileNetV2, VGG16)

In [ ]:
data224 = tf.keras.utils.image_dataset_from_directory('CommonVoice/mfcc/mfcc_images', batch_size=32 , image_size = (224,224) ) 
data224 = data224.map(lambda x,y: (x/255, y))

Image size 299x299 (InceptionV3, Xception)

In [ ]:
data299 = tf.keras.utils.image_dataset_from_directory('CommonVoice/mfcc/mfcc_images', batch_size=32 , image_size = (299,299))
data299 = data299.map(lambda x,y: (x/255, y))

Split Into Train, Test, Validation Sets

In [ ]:
def split_data(data):

    train_size = int(len(data)*.8)
    val_size = int(len(data)*.1)
    test_size = int(len(data)*.1)
    train = data.take(train_size)
    val = data.skip(train_size).take(val_size)
    test = data.skip(train_size+val_size).take(test_size)

    return {
        "train": train,
        "val": val,
        "test": test,
    }

In [ ]:
data224 = split_data(data224)

In [ ]:
data256 = split_data(data256)

In [ ]:
data299 = split_data(data299)

<h3>InceptionV3</h3>

Training

In [ ]:
InceptionV3_model = InceptionV3(include_top=False, input_shape=(299, 299, 3))

for layer in InceptionV3_model.layers:
    layer.trainable = False

InceptionV3_model = Sequential([
    InceptionV3_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
InceptionV3_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
start_train_time = time.time()
IV3_hist = InceptionV3_model.fit(data299['train'], epochs=1, validation_data=data299['val'], callbacks=[tensorboard_callback,early_stopping])
end_train_time = time.time()
IV3_training_time = end_train_time - start_train_time
print(f"Training time: {IV3_training_time:.4f} seconds")

Test

In [ ]:
plot_metrics(model_title="InceptionV3", history=IV3_hist, test_set=data224['test'], model=InceptionV3_model)

Export

In [ ]:
os.makedirs('Models', exist_ok=True)
InceptionV3_model.export('Models/InceptionV3')

<h2>Xception</h2>

Training

In [ ]:
Xception_model = Xception(include_top=False, input_shape=(299, 299, 3))

for layer in Xception_model.layers:
    layer.trainable = False

Xception_model = Sequential([
    Xception_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compile the model
Xception_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
start_train_time = time.time()
Xcp_hist = Xception_model.fit(data299['train'], epochs=1, validation_data=data299['val'], callbacks=[tensorboard_callback,early_stopping])
end_train_time = time.time()
Xcp_training_time = end_train_time - start_train_time
print(f"Training time: {Xcp_training_time:.4f} seconds")

Testing

In [ ]:
plot_metrics(model_title="Xception", history=Xcp_hist, test_set=data224['test'], model=Xception_model)

Export

In [ ]:
os.makedirs('Models', exist_ok=True)
Xception_model.export('Models/Xception')

<h2>ResNet50</h2>

Training

In [ ]:
resnet50_model = ResNet50(include_top=False, input_shape=(224, 224, 3))

for layer in resnet50_model.layers:
    layer.trainable = False

# build the entire model
resnet50_model = tf.keras.Sequential([
    resnet50_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')  
])

logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
resnet50_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

start_train_time = time.time()
R50_hist = resnet50_model.fit(data224['train'], epochs=1, validation_data=data224['val'], callbacks=[tensorboard_callback,early_stopping])
end_train_time = time.time()
R50_training_time = end_train_time - start_train_time
print(f"Training time: {R50_training_time:.4f} seconds")

Testing

In [ ]:
plot_metrics(model_title="ResNet50", history=R50_hist, test_set=data224['test'], model=resnet50_model)

Export

In [ ]:
os.makedirs('Models', exist_ok=True)
resnet50_model.export('Models/ResNet50')

<h2>MobileNetV2</h2>

Training

In [ ]:
mobilenetv2_model = MobileNetV2(include_top=False, input_shape=(224,224,3))

for layer in mobilenetv2_model.layers:
    layer.trainable = False

# Create new model with MobileNetV2 base and custom classification head
mobilenetv2_model = Sequential([
    mobilenetv2_model,
    GlobalAveragePooling2D(),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')  # Adjust num_classes according to your dataset
])

# Compile the model
mobilenetv2_model.compile(optimizer='adam',loss=tf.losses.BinaryCrossentropy(),metrics=['accuracy'])
logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

start_train_time = time.time()
MnV2_hist = mobilenetv2_model.fit(data224['train'], epochs=1, validation_data=data224['val'], callbacks=[tensorboard_callback,early_stopping])
end_train_time = time.time()
MnV2_training_time = end_train_time - start_train_time
print(f"Training time: {MnV2_training_time:.4f} seconds")

Testing

In [ ]:
plot_metrics(model_title="MobileNetV2", history=MnV2_hist, test_set=data224['test'], model=mobilenetv2_model)

Export

In [ ]:
os.makedirs('Models', exist_ok=True)
mobilenetv2_model.export('Models/MobileNetV2')

<h2>VGG16</h2>

Training

In [ ]:
vgg16_model = VGG16(input_shape =(224,224,3),include_top = False)

for layer in vgg16_model.layers:
    layer.trainable = False

vgg16_model = Sequential([
    vgg16_model,
    Flatten(),
    Dense(512,activation='relu'),
    Dropout(0.5),
    Dense(1,activation='sigmoid'),    
])

vgg16_model.compile(optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.0001), loss = 'binary_crossentropy',metrics = ['accuracy'])
logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

start_train_time = time.time()
vgg16_hist = vgg16_model.fit(data224['train'], epochs=1, validation_data=data224['val'], callbacks=[tensorboard_callback,early_stopping])
end_train_time = time.time()
VGG16_training_time = end_train_time - start_train_time
print(f"Training time: {VGG16_training_time:.4f} seconds")

Testing

In [ ]:
plot_metrics(model_title="VGG16", history=vgg16_hist, test_set=data224['test'], model=vgg16_model)

Export

In [ ]:
os.makedirs('Models', exist_ok=True)
vgg16_model.export('Models/VGG16')

<h2>CNN

Training

In [ ]:
CNN_model = tf.keras.Sequential([
    Conv2D(16, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    MaxPool2D(pool_size=2),
    BatchNormalization(),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPool2D(pool_size=2),
    BatchNormalization(),
    Conv2D(16, (3, 3), activation='relu'),
    MaxPool2D(pool_size=2),
    BatchNormalization(),
    Flatten(),
    Dropout(0.5),
    Dense(256, activation="relu"),
    Dropout(0.5),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

In [ ]:
CNN_model.compile(optimizer='adam',loss=tf.losses.BinaryCrossentropy(),metrics=['accuracy'])
logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=logdir)
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

start_train_time = time.time()
CNN_hist = CNN_model.fit(data256['train'], epochs=1, validation_data=data256['val'], callbacks=[tensorboard_callback,early_stopping])
end_train_time = time.time()
CNN_training_time = end_train_time - start_train_time
print(f"Training time: {CNN_training_time:.4f} seconds")


Testing

In [ ]:
plot_metrics(model_title="CNN", history=CNN_hist, test_set=data224['test'], model=CNN_model)

Export

In [ ]:
os.makedirs('Models', exist_ok=True)
CNN_model.export('Models/CNN')